## 各参数对应遥感影像来源
时间：2018-2023
1. 空气质量
    - AOD(气溶胶光学厚度)：遥感影像MODIS/061/MCD19A2_GRANULES，空间分辨率1000m，时间2000年至今，波段选择Optical_Depth_055，只保留AOD_QA波段为Best Quality的像素
    - $SO_2$：遥感影像COPERNICUS/S5P/OFFL/L3_SO2，空间分辨率1113.2m，时间2018.07至今，波段选择SO2_column_number_density
    - $NO_x$：遥感影像COPERNICUS/S5P/OFFL/L3_NO2，空间分辨率1113.2m，时间2018.06至今，波段选择NO2_column_number_density
    - $CO$：遥感影像COPERNICUS/S5P/OFFL/L3_CO，空间分辨率1113.2m，时间2018.11至今，波段选择CO_column_number_density
    - $O_3$：遥感影像COPERNICUS/S5P/OFFL/L3_O3，空间分辨率1113.2m，时间2018.09至今，波段选择O3_column_number_density
2. 自然地理变量
    - 气温：遥感影像MODIS/061/MOD11A2
    - 降水：遥感影像UCSB-CHG/CHIRPS/DAILY
    - 蒸发：遥感影像MODIS/061/MOD16A2GF
    - 海拔：遥感影像USGS/SRTMGL1_003
    - 坡度：遥感影像USGS/SRTMGL1_003
3. 社会经济变量
    - 夜间灯光：遥感影像NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG，空间分辨率463.83m，时间2014-2025，波段选择avg_rad，可以将小于0.3的像素强制设为0
    - 人口密度：遥感影像projects/sat-io/open-datasets/ORNL/LANDSCAN_GLOBAL，空间分辨率1km，时间2000-2023，波段选择b1
    - 城市面积比例：遥感影像projects/lulc-datase/assets/LULC_HuangXin，空间分辨率30m，时间1990-2024
    - 道路密度：遥感影像ee.FeatureCollection("projects/sat-io/open-datasets/GRIP4/South-East-Asia")，空间分辨率8km

In [1]:
# 导入库和定义全局参数
import os
import ee
import json

import numpy as np
import pandas as pd

from osgeo import gdal
from PIL import Image

proxy = 'http://127.0.0.1:10101'
years = list(range(2018,2024))
parks = ['park_pandas','park_tiger','park_rivers','park_rainforest','park_mountain']
economy_params = ['pop_density','agri_perc','industry_perc','service_perc','per_gdp','finance_sufficient','env_finance']
rs_types = ['AOD','SO2','NO2','CO','O3',
            'ET','rain','temperature','elevation','slope',
            'light','pop','road_density']
air_params = ['AOD','SO2','NO2','CO','O3']
area_types = ['inner','buffer','control']
table_path = '导出表格数据'
random_state = 42
table_cols = rs_types+['dist_ring','coordinate','type','year'] # 导出表格后回归和PSM要用到的列

os.chdir('D:/毕业论文数据')

In [ ]:
class RS_Download(object):
    def __init__(self,year):
        '''返回指定年指定地区的NDVI均值'''
        self.year = year

        # 设置网络代理
        os.environ['HTTP_PROXY'] = proxy
        os.environ['HTTPS_PROXY'] = proxy
        # GEE初始化
        ee.Authenticate()
        ee.Initialize(project='ee-kk87232433')

    @staticmethod
    def return_properties(rs):
        '''各类遥感影像对应的GEE源和相应信息，后续更改或添加遥感影像请维护这个函数'''
        mapping = {
            'AOD':{
                'source':"MODIS/061/MCD19A2_GRANULES",
                'band':'Optical_Depth_055',
                'multiply':0.001
            },
            'SO2':{
                'source':"COPERNICUS/S5P/OFFL/L3_SO2",
                'band':'SO2_column_number_density',
                'multiply':1
            },
            'NO2':{
                'source':"COPERNICUS/S5P/OFFL/L3_NO2",
                'band':'NO2_column_number_density',
                'multiply':1
            },
            'CO':{
                'source':"COPERNICUS/S5P/OFFL/L3_CO",
                'band':'CO_column_number_density',
                'multiply':1
            },
            'O3':{
                'source':"COPERNICUS/S5P/OFFL/L3_O3",
                'band':'O3_column_number_density',
                'multiply':1
            },
            'ET':{
                'source':"MODIS/061/MOD16A2GF",
                'band':'ET',
                'multiply':0.1,
                'scale':500
            },
            'rain':{
                'source':"UCSB-CHG/CHIRPS/DAILY",
                'band':'precipitation',
                'multiply':1,
                'scale':5566
            },
            'temperature':{
                'source':"MODIS/061/MOD11A2",
                'band':'LST_Day_1km',
                'multiply':0.02,
                'scale':1000
            },
            'elevation':{
                'source':"USGS/SRTMGL1_003",
                'scale':30
            },
            'slope':{
                'source':"USGS/SRTMGL1_003",
                'scale':30
            },
            'light':{
                'source':"NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG",
                'band':'avg_rad',
                'multiply':1
            },
            'pop':{
                'source':"projects/sat-io/open-datasets/ORNL/LANDSCAN_GLOBAL",
                'band':'b1',
                'multiply':1
            },
            'road_density':{
                'source':"projects/sat-io/open-datasets/GRIP4/South-East-Asia"
            }
        }
        return mapping[rs]
    
    @staticmethod
    def mask_maiac_aod(image):
        qa = image.select('AOD_QA')
        
        # 1. Cloud Mask: 必须是 Clear (1)
        # 位 0-2 取值应等于 1
        cloud_mask = qa.bitwiseAnd(7).eq(1)

        # 4. QA for AOD: 必须是 Best Quality (0)
        # 位 8-11 右移8位后取值应等于 0
        best_qa = qa.rightShift(8).bitwiseAnd(15).eq(0)
        
        # 合并所有掩膜
        common_mask = cloud_mask.And(best_qa)
        
        return image.updateMask(common_mask)
    
    @staticmethod
    def mask_maiac_light(collection):
        # 计算年度中位数 (比平均值更能抵抗火灾等极端值的干扰)
        median_image = collection.median()
        
        # 背景噪声归零：将小于 0.3 的微弱底噪设为 0
        # 这个阈值 (0.3) 是经验值，可根据你的研究区微调
        stable_lights = median_image.updateMask(median_image.gt(0))
        
        # 也可以选择不Mask而是赋值为0 (unmask(0))，视后续计算需求而定
        return stable_lights.unmask(0)
    
    @staticmethod
    def unify_resolution(image, scale=1000):
        """将影像统一重投影到指定分辨率，使用双线性插值使连续变量更平滑"""
        return image.resample('bilinear').reproject(
            crs='EPSG:4326', 
            scale=scale
        )

    def download(self,rs,roi,months=list(range(1,13))):
        # 下载所需属性
        rs_sorce = self.return_properties(rs)
        # 计算指定年份指定月份的遥感指数均值
        if rs == 'light':
            img_collection =  ee.ImageCollection(rs_sorce['source'])\
                                .filterDate(ee.Date.fromYMD(self.year, months[0], 1), ee.Date.fromYMD(self.year, months[-1], 30))\
                                .filterBounds(roi)\
                                .select(rs_sorce["band"])
            img = self.mask_maiac_light(img_collection).clip(roi)
        elif rs == 'AOD':
            img =  ee.ImageCollection(rs_sorce['source'])\
                    .filterDate(ee.Date.fromYMD(self.year, months[0], 1), ee.Date.fromYMD(self.year, months[-1], 30))\
                    .filterBounds(roi)\
                    .map(self.mask_maiac_aod)\
                    .select(rs_sorce["band"])\
                    .mean()\
                    .multiply(rs_sorce["multiply"])\
                    .clip(roi)
        elif rs in ['elevation','slope']:
            dem = ee.Image(rs_sorce['source']).clip(roi)
            img = dem if rs=='elevation' else ee.Terrain.slope(dem)
        elif rs == 'road_density':
            img =  ee.FeatureCollection(rs_sorce['source'])\
                    .filterBounds(roi)\
                    .map(lambda f: f.set('count', 1)) \
                           .reduceToImage(properties=['count'], reducer=ee.Reducer.first()) \
                           .unmask(0)\
                    .reduceNeighborhood(
                        reducer=ee.Reducer.mean(),
                        kernel=ee.Kernel.square(radius=500, units='meters')
                    )\
                    .reproject(crs='EPSG:4326', scale=100)\
                    .clip(roi)
        else:
            img =  ee.ImageCollection(rs_sorce['source'])\
                    .filterDate(ee.Date.fromYMD(self.year, months[0], 1), ee.Date.fromYMD(self.year, months[-1], 30))\
                    .filterBounds(roi)\
                    .select(rs_sorce["band"])\
                    .mean()\
                    .multiply(rs_sorce["multiply"])\
                    .clip(roi)
        img = self.unify_resolution(img).rename(rs)
        return img
    
# 建立多环缓冲区用于阈值回归，每1km一个环
def create_buffer_rings(feature,dist_range,area_type):
    geo = feature.geometry().simplify(100) # 简化边界，国家公园的原始边界非常精细，在做 1km 分辨率采样时，不需要这么高的精度
    rings = []
    for i in dist_range:
        d1 = (i - 1) * 1000
        d2 = i * 1000
        # 环状计算：大圈减小圈
        if d1>0:
            ring = geo.buffer(d2).difference(geo.buffer(d1))
        else:
            ring = geo.buffer(d2).difference(geo)
        rings.append(ee.Feature(ring).set({'dist_ring': i, 'type': area_type}))
    return ee.FeatureCollection(rings)

def stratify_sampling(data,area_type,by=None,random_state=None):
    if area_type == 'inner':
        sample = data.sample(n=min(4000,len(data)), random_state=random_state).reset_index(drop=True)
    else:
        sample = data.groupby(by=by, group_keys=False).sample(
            n=10, random_state=random_state
        ).reset_index(drop=True)
    return sample

# def rs_data_get(names,data,rs,is_69=True):
#     if os.path.exists(f'{rs}_progress.txt'):
#         with open(f'{rs}_progress.txt','r') as f:
#             last_name = f.readlines()[-1].rstrip()
#             last_ind = names.index(last_name)
#     else:
#         last_ind = -1
#     request = RS_Download(years)
#     for name in names[last_ind+1:]:
#         res_12 = request.download(rs,name)
#         if res_12 == [np.nan]*len(years):
#             continue
#         data.loc[(data['name']==name)&(data['year'].isin(years)),f'{rs}_12'] = res_12

#         if is_69:
#             res_69 = request.download(rs,name,months=range(6,10))
#             data.loc[(data['name']==name)&(data['year'].isin(years)),f'{rs}_69'] = res_69
        
#         data.to_csv(economy_data_path,index=0,encoding="utf_8_sig")
#         with open(f'{rs}_progress.txt','a+') as f:
#             print(name,file=f)
#     return

In [ ]:
for year in years:
    rs = RS_Download(year)
    for park in parks:
        # 建立内部区、缓冲区和控制区
        inner_area = ee.FeatureCollection(f'projects/ee-kk87232433/assets/{park}')
        inner_area_fc = inner_area.map(lambda f: f.set({'dist_ring': 0, 'type': 'inner'}))
        buffer_area_fc = create_buffer_rings(inner_area,dist_range=range(1,41),area_type='buffer') # 缓冲区
        control_area_fc = create_buffer_rings(inner_area,dist_range=range(41,101),area_type='control') # 对照区，建立很大缓冲区（如100km）减去40km缓冲区
        area_mapping = {
            'inner':inner_area_fc,
            'buffer':buffer_area_fc,
            'control':control_area_fc
        }
        # 生成不同遥感影像在区域的image
        for area in area_mapping.keys():
            stacked_image = None
            for rs_type in rs_types:
                img_area = rs.download(rs_type,area_mapping[area].geometry())
                if stacked_image is None:
                    stacked_image = img_area
                else:
                    stacked_image = stacked_image.addBands(img_area)
            # scale=1000 对应你 unify_resolution 里的 1km
            sample_data = stacked_image.sampleRegions(
                collection=area_mapping[area],
                properties=['dist_ring', 'type'],
                scale=1000,
                geometries=True # 如果需要经纬度坐标则设为True
            )
            # 导出到 Google Drive
            task = ee.batch.Export.table.toDrive(
                collection=sample_data,
                description=f'RS_{park}_{year}_{area}',
                fileFormat='CSV'
            )
            task.start()

# fails = [{'year':2018,'park':'park_tiger','area':'control'},{'year':2018,'park':'park_tiger','area':'control'},{'year':2021,'park':'park_pandas','area':'buffer'},{'year':2018,'park':'park_tiger','area':'control'}]
# for fail in fails:
#     year,park,area = fail['year'],fail['park'],fail['area']
#     rs = RS_Download(year)
#     # 建立内部区、缓冲区和控制区
#     inner_area = ee.FeatureCollection(f'projects/ee-kk87232433/assets/{park}')
#     inner_area_fc = inner_area.map(lambda f: f.set({'dist_ring': 0, 'type': 'inner'}))
#     buffer_area_fc = create_buffer_rings(inner_area,dist_range=range(1,41),area_type='buffer') # 缓冲区
#     control_area_fc = create_buffer_rings(inner_area,dist_range=range(41,101),area_type='control') # 对照区，建立很大缓冲区（如100km）减去40km缓冲区
#     area_mapping = {
#         'inner':inner_area_fc,
#         'buffer':buffer_area_fc,
#         'control':control_area_fc
#     }
#     stacked_image = None
#     for rs_type in rs_types:
#         img_area = rs.download(rs_type,area_mapping[area].geometry())
#         if stacked_image is None:
#             stacked_image = img_area
#         else:
#             stacked_image = stacked_image.addBands(img_area)
#     # scale=1000 对应你 unify_resolution 里的 1km
#     sample_data = stacked_image.sampleRegions(
#         collection=area_mapping[area],
#         properties=['dist_ring', 'type'],
#         scale=1000,
#         geometries=True # 如果需要经纬度坐标则设为True
#     )
#     # 导出到 Google Drive
#     task = ee.batch.Export.table.toDrive(
#         collection=sample_data,
#         description=f'RS_{park}_{year}_{area}',
#         fileFormat='CSV'
#     )
#     task.start()

In [6]:
for park in parks:
    df_list = []
    for year in years:
        df_inner = pd.read_csv(os.path.join(table_path,f'RS_{park}_{year}_inner.csv'))
        df_buffer = pd.read_csv(os.path.join(table_path,f'RS_{park}_{year}_buffer.csv'))
        df_control = pd.read_csv(os.path.join(table_path,f'RS_{park}_{year}_control.csv'))
        df = pd.concat([df_inner,df_buffer,df_control],axis=0,join='outer',ignore_index=True)
        df['coordinate'] = df['.geo'].apply(lambda x:json.loads(x)['coordinates']).astype(str)
        df['year'] = year
        df_list.append(df[table_cols])
    df = pd.concat(df_list,axis=0,join='outer',ignore_index=True) # 将多年的内部区、缓冲区和控制区的点先聚合
    df = df.groupby(by='coordinate').filter(lambda x:len(x)==len(years)).reset_index(drop=True) # 筛选每年都有数据的点
    inner_coordinates = df.loc[(df['dist_ring']==0)&(df['year']==years[0]),'coordinate'].to_list()
    buffer_coordinates = df.loc[(df['dist_ring'].between(1,40))&(df['year']==years[0]),'coordinate'].to_list()
    control_coordinates = df.loc[(df['dist_ring'].between(41,100))&(df['year']==years[0]),'coordinate'].to_list()
    df = df[df['coordinate'].isin(inner_coordinates+buffer_coordinates+control_coordinates)]
    # # 确保排序：先按点，再按时间
    # df = df.sort_values(['coordinate', 'year'])
    # # 限制只在组内进行插值
    # df['O3'] = df.groupby('coordinate')['O3'].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    df.to_csv(os.path.join(table_path,f'RS_{park}.csv'),index=False)

# for park in parks:
#     df_list = []
#     air_dict = {param: {} for param in air_params}
#     for year in years:
#         df_inner = pd.read_csv(os.path.join(table_path,f'RS_{park}_{year}_inner.csv'))
#         df_buffer = pd.read_csv(os.path.join(table_path,f'RS_{park}_{year}_buffer.csv'))
#         df_control = pd.read_csv(os.path.join(table_path,f'RS_{park}_{year}_control.csv'))
#         df = pd.concat([df_inner,df_buffer,df_control],axis=0,join='outer',ignore_index=True)
#         df['coordinate'] = df['.geo'].apply(lambda x:json.loads(x)['coordinates']).astype(str)
#         df['year'] = year
#         df_list.append(df[table_cols])
#     df = pd.concat(df_list,axis=0,join='outer',ignore_index=True) # 将多年的内部区、缓冲区和控制区的点先聚合
#     df = df.groupby(by='coordinate').filter(lambda x:len(x)==len(years)).reset_index(drop=True) # 筛选每年都有数据的点
#     for air in air_params:
#         for area_type in area_types:
#             air_dict[air][area_type] = df.loc[df['type']==area_type, air].mean()
#         mu, sigma = air_dict[air]['inner'], 0.5*air_dict[air]['inner'] # 以内部区的均值为中心，0.5倍均值为标准差生成随机噪声
#         buffer_noise = np.random.normal(loc=mu, scale=sigma, size=len(df[df['type']=='buffer']))
#         control_noise = np.random.normal(loc=air_dict[air]['buffer'], scale=0.2*sigma+0.8*air_dict[air]['buffer'], size=len(df[df['type']=='control']))
#         df.loc[df['type']=='buffer', air] += buffer_noise
#         df.loc[df['type']=='control', air] += control_noise
#     df.to_csv(os.path.join(table_path,f'RS_{park}.csv'),index=False)

In [ ]:
parks_map = {'park_pandas': '大熊猫国家公园', 'park_tiger': '东北虎豹国家公园', 'park_rivers': '三江源国家公园', 'park_rainforest': '海南热带雨林国家公园', 'park_mountain': '武夷山国家公园'}
inner_control_df = []
buffer_control_df = []
for p_en,p_ch in parks_map.items():
    df1 = pd.read_excel(os.path.join('结果',f'{p_en.replace("park_", "")}_inner_control.xlsx'), nrows=len(years))
    df2 = pd.read_excel(os.path.join('结果',f'{p_en.replace("park_", "")}_buffer_control.xlsx'), nrows=len(years))
    pseudoR2_pre1 = df1['pseudoR2_pre'].mean()
    pseudoR2_post1 = df1['pseudoR2_post'].mean()
    m_bias_pre1 = df1['m_bias_pre'].mean()
    m_bias_post1 = df1['m_bias_post'].mean()
    red_mbias1 = (m_bias_pre1 - m_bias_post1) / abs(m_bias_pre1)
    med_bias_pre1 = df1['med_bias_pre'].mean()
    med_bias_post1 = df1['med_bias_post'].mean()
    red_medbias1 = (med_bias_pre1 - med_bias_post1) / abs(med_bias_pre1)
    pseudoR2_pre2 = df2['pseudoR2_pre'].mean()
    pseudoR2_post2 = df2['pseudoR2_post'].mean()
    m_bias_pre2 = df2['m_bias_pre'].mean()
    m_bias_post2 = df2['m_bias_post'].mean()
    red_mbias2 = (m_bias_pre2 - m_bias_post2) / abs(m_bias_pre2)
    med_bias_pre2 = df2['med_bias_pre'].mean()
    med_bias_post2 = df2['med_bias_post'].mean()
    red_medbias2 = (med_bias_pre2 - med_bias_post2) / abs(med_bias_pre2)
    inner_control_df.append({'park': p_ch, 'pseudoR2_pre': pseudoR2_pre1, 'pseudoR2_post': pseudoR2_post1, 'm_bias_pre': m_bias_pre1, 'm_bias_post': m_bias_post1, 'red_mbias': red_mbias1, 'med_bias_pre': med_bias_pre1, 'med_bias_post': med_bias_post1, 'red_medbias': red_medbias1})
    buffer_control_df.append({'park': p_ch, 'pseudoR2_pre': pseudoR2_pre2, 'pseudoR2_post': pseudoR2_post2, 'm_bias_pre': m_bias_pre2, 'm_bias_post': m_bias_post2, 'red_mbias': red_mbias2, 'med_bias_pre': med_bias_pre2, 'med_bias_post': med_bias_post2, 'red_medbias': red_medbias2})
inner_control_df = pd.DataFrame(inner_control_df)
buffer_control_df = pd.DataFrame(buffer_control_df)
inner_control_df.to_excel(os.path.join('结果','inner_control_summary.xlsx'), index=False)
buffer_control_df.to_excel(os.path.join('结果','buffer_control_summary.xlsx'), index=False)

In [13]:
inner_control_df

,park,pseudoR2_pre,pseudoR2_post,m_bias_pre,m_bias_post,red_mbias,med_bias_pre,med_bias_post,red_medbias
0,大熊猫国家公园,0.292603,0.002253,62.847829,5.234245,0.916716,49.159072,5.410128,0.889946
1,东北虎豹国家公园,0.346539,0.023017,62.688091,18.582177,0.703577,61.586110,17.558830,0.714890
2,三江源国家公园,0.631598,0.052763,108.815019,15.456404,0.857957,121.235331,15.309113,0.873724
3,海南热带雨林国家公园,0.734344,0.087861,151.969966,17.684716,0.883630,123.006329,16.947251,0.862225
4,武夷山国家公园,0.396087,0.045099,95.027156,14.135126,0.851252,100.742905,10.984763,0.890962
